# Module 7.1: Brightness & Contrast Enhancement Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_07_RL_For_Image_Enhancement/01_Brightness_Contrast_Agent/brightness_contrast_agent.ipynb)

---

## Learning Objectives

By the end of this notebook you will be able to:

1. **Formulate** image brightness/contrast adjustment as a Markov Decision Process (MDP)
2. **Implement** a Gymnasium-compatible image enhancement environment
3. **Build and train** a Deep Q-Network (DQN) agent that learns to restore degraded images
4. **Evaluate** image quality using PSNR and SSIM metrics
5. **Visualise** training dynamics and before/after enhancement results

---

## 1 — Mathematical Foundation

### 1.1 Brightness and Contrast Model

Given an input image $I(x,y)$ with pixel intensities in $[0,1]$, a linear brightness–contrast transformation is:

$$I'(x,y) = \alpha \cdot I(x,y) + \beta$$

where $\alpha > 0$ controls **contrast** (gain) and $\beta \in [-1,1]$ controls **brightness** (bias). The output is clipped to $[0,1]$.

### 1.2 MDP Formulation

We cast image enhancement as a finite-horizon MDP $\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

| Component | Definition |
|-----------|------------|
| **State** $s_t$ | Current image tensor $I_t \in \mathbb{R}^{H \times W \times C}$ |
| **Action** $a_t$ | Discrete adjustment: $(\Delta\alpha, \Delta\beta)$ from a predefined set |
| **Transition** $P$ | Deterministic: $I_{t+1} = \text{clip}(\alpha_t \cdot I_t + \beta_t)$ |
| **Reward** $r_t$ | Change in image quality: $r_t = Q(I_{t+1}, I^*) - Q(I_t, I^*)$ |
| **Discount** $\gamma$ | $0.99$ |

### 1.3 Image Quality Metrics

**Peak Signal-to-Noise Ratio (PSNR):**

$$\text{PSNR}(I, I^*) = 10 \cdot \log_{10}\!\left(\frac{\text{MAX}^2}{\text{MSE}(I, I^*)}\right), \quad \text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(I_i - I^*_i)^2$$

**Structural Similarity Index (SSIM):**

$$\text{SSIM}(I, I^*) = \frac{(2\mu_I \mu_{I^*} + C_1)(2\sigma_{II^*} + C_2)}{(\mu_I^2 + \mu_{I^*}^2 + C_1)(\sigma_I^2 + \sigma_{I^*}^2 + C_2)}$$

where $C_1 = (k_1 L)^2$, $C_2 = (k_2 L)^2$, $L = 1$ (dynamic range), $k_1 = 0.01$, $k_2 = 0.03$.

**Combined reward signal:**

$$r_t = \lambda_1 \,\Delta\text{PSNR}_t + \lambda_2 \,\Delta\text{SSIM}_t$$

### 1.4 Deep Q-Network

The agent learns an action-value function $Q(s,a;\theta)$ via a CNN. Training minimises the Huber loss:

$$\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s')\sim\mathcal{D}}\left[\text{Huber}\!\left(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)\right)\right]$$

where $\theta^-$ are the periodically-updated target network weights and $\mathcal{D}$ is the experience replay buffer.

## 2 — Environment Setup

## Deep Derivation: Image Quality Metrics for RL Reward

### Step 1: Mean Squared Error (MSE)
$$\text{MSE} = \frac{1}{MN}\sum_{i=0}^{M-1}\sum_{j=0}^{N-1} [I(i,j) - \hat{I}(i,j)]^2$$

### Step 2: PSNR Derivation
$$\text{PSNR} = 10 \cdot \log_{10}\left(\frac{MAX_I^2}{\text{MSE}}\right) = 20 \cdot \log_{10}\left(\frac{MAX_I}{\sqrt{\text{MSE}}}\right)$$

**Step-by-step:**
1. Let $MAX_I = 255$ for 8-bit images
2. $\text{PSNR} = 10\log_{10}(255^2/\text{MSE})$
3. $= 10\log_{10}(255^2) - 10\log_{10}(\text{MSE})$
4. $= 20\log_{10}(255) - 10\log_{10}(\text{MSE})$
5. $\approx 48.13 - 10\log_{10}(\text{MSE})$

**Interpretation:** +6 dB ≈ halving the error, +20 dB ≈ 10× less error.

### Step 3: SSIM Full Derivation
SSIM combines three components:

**Luminance:** $l(x,y) = \frac{2\mu_x\mu_y + C_1}{\mu_x^2 + \mu_y^2 + C_1}$

**Contrast:** $c(x,y) = \frac{2\sigma_x\sigma_y + C_2}{\sigma_x^2 + \sigma_y^2 + C_2}$

**Structure:** $s(x,y) = \frac{\sigma_{xy} + C_3}{\sigma_x\sigma_y + C_3}$

**Combined:** $\text{SSIM}(x,y) = l(x,y)^\alpha \cdot c(x,y)^\beta \cdot s(x,y)^\gamma$

With $\alpha=\beta=\gamma=1$ and $C_3 = C_2/2$:
$$\text{SSIM}(x,y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

**Why $C_1, C_2$?** Stability constants to avoid division by zero:
$C_1 = (K_1 \cdot L)^2$, $C_2 = (K_2 \cdot L)^2$ where $L=255$, $K_1=0.01$, $K_2=0.03$

### Step 4: Statistics Computation (11×11 Gaussian window)
$$\mu_x = \sum_i w_i x_i, \quad \sigma_x^2 = \sum_i w_i(x_i - \mu_x)^2, \quad \sigma_{xy} = \sum_i w_i(x_i-\mu_x)(y_i-\mu_y)$$

### Step 5: RL Reward Design
$$r_t = \alpha \cdot \Delta\text{PSNR} + \beta \cdot \Delta\text{SSIM}$$

where $\Delta\text{PSNR} = \text{PSNR}(I_{t+1}, I_{target}) - \text{PSNR}(I_t, I_{target})$

**Why both?** PSNR measures pixel accuracy; SSIM measures perceptual quality. An agent that only maximizes PSNR might produce blurry images (low MSE but bad structure).

In [ ]:
!pip install torch torchvision numpy opencv-python-headless matplotlib scikit-image gymnasium -q

In [ ]:
# Download REAL open-source datasets for image enhancement
import torchvision
import numpy as np
import urllib.request
import os

# CIFAR-10 real photographs (our base images to enhance)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = np.array([np.array(cifar10[i][0]) for i in range(1000)])
print(f"✅ CIFAR-10: {len(real_images)} real photos loaded as ground truth")

# BSD68 denoising benchmark (68 real test images)
bsd_url = "https://raw.githubusercontent.com/clausmichele/CBSD68-dataset/master/CBSD68/original/"
os.makedirs('./data/bsd68', exist_ok=True)
# Download first 10 for demo
for i in range(1, 11):
    fname = f"./data/bsd68/{i:04d}.png"
    if not os.path.exists(fname):
        try:
            urllib.request.urlretrieve(f"{bsd_url}{i:04d}.png", fname)
        except:
            pass
print(f"✅ BSD68 benchmark: downloading real denoising test images")
print(f"   These are REAL photographs used in denoising research papers!")

In [ ]:
import os, random, math, collections
from typing import Optional, Tuple

import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
import gymnasium as gym
from gymnasium import spaces

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3 — Synthetic Dataset Generation

We create a small set of synthetic images with geometric patterns and textures, then degrade them with random brightness/contrast distortions.

In [ ]:
IMG_SIZE = 64
NUM_IMAGES = 50


def generate_synthetic_image(size: int = IMG_SIZE) -> np.ndarray:
    """Generate a synthetic RGB image with geometric patterns."""
    img = np.zeros((size, size, 3), dtype=np.float32)
    num_shapes = random.randint(3, 8)
    for _ in range(num_shapes):
        colour = np.array([random.random(), random.random(), random.random()], dtype=np.float32)
        shape_type = random.choice(["rect", "circle", "line"])
        if shape_type == "rect":
            x1, y1 = random.randint(0, size - 10), random.randint(0, size - 10)
            x2, y2 = x1 + random.randint(5, size // 2), y1 + random.randint(5, size // 2)
            img[y1:y2, x1:x2] = colour
        elif shape_type == "circle":
            cx, cy = random.randint(10, size - 10), random.randint(10, size - 10)
            r = random.randint(3, size // 4)
            yy, xx = np.ogrid[:size, :size]
            mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
            img[mask] = colour
        else:
            img_u8 = (img * 255).astype(np.uint8)
            pt1 = (random.randint(0, size), random.randint(0, size))
            pt2 = (random.randint(0, size), random.randint(0, size))
            cv2.line(img_u8, pt1, pt2, (int(colour[0]*255), int(colour[1]*255), int(colour[2]*255)), 2)
            img = img_u8.astype(np.float32) / 255.0

    # Add smooth gradient background
    grad = np.linspace(0.1, 0.4, size).reshape(1, -1, 1).astype(np.float32)
    bg_mask = (img.sum(axis=2) == 0)
    img[bg_mask] = np.broadcast_to(grad, (size, size, 3))[bg_mask]
    return np.clip(img, 0.0, 1.0)


def degrade_image(img: np.ndarray) -> Tuple[np.ndarray, float, float]:
    """Apply random brightness/contrast degradation."""
    alpha = random.uniform(0.4, 0.7)   # reduce contrast
    beta = random.uniform(-0.3, 0.3)   # shift brightness
    degraded = np.clip(alpha * img + beta, 0.0, 1.0).astype(np.float32)
    return degraded, alpha, beta


# Build dataset
dataset = []
for _ in range(NUM_IMAGES):
    target = generate_synthetic_image()
    degraded, alpha, beta = degrade_image(target)
    dataset.append({"target": target, "degraded": degraded, "alpha": alpha, "beta": beta})

# Visualise samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    axes[0, i].imshow(dataset[i]["target"])
    axes[0, i].set_title(f"Target {i}")
    axes[0, i].axis("off")
    axes[1, i].imshow(dataset[i]["degraded"])
    axes[1, i].set_title(f"Degraded (α={dataset[i]['alpha']:.2f})")
    axes[1, i].axis("off")
plt.suptitle("Synthetic Image Pairs: Target (top) vs Degraded (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

## Deep Derivation: Histogram Equalization as Optimal Transport

### Step 1: Histogram Equalization — CDF Transform

For a grayscale image with intensity PDF $p(r)$, $r \in [0, 1]$, the transform:

$$s = T(r) = \int_0^r p(w) \, dw = \text{CDF}(r)$$

produces an output with **uniform distribution** $p_s(s) = 1$ on $[0, 1]$.

### Step 2: Proof of Uniformity

Using the change-of-variable formula for probability densities:

$$p_s(s) = p_r(r) \left|\frac{dr}{ds}\right|$$

Since $s = \text{CDF}(r)$, we have $\frac{ds}{dr} = p_r(r)$, therefore $\frac{dr}{ds} = \frac{1}{p_r(r)}$.

$$p_s(s) = p_r(r) \cdot \frac{1}{p_r(r)} = 1 \quad \blacksquare$$

### Step 3: Histogram Equalization as 1D Optimal Transport

The **1D Wasserstein distance** (Earth Mover's Distance) between distributions $p$ and $q$:

$$W_1(p, q) = \int_0^1 |F_p^{-1}(t) - F_q^{-1}(t)| \, dt$$

where $F^{-1}$ is the quantile function (inverse CDF).

**The optimal transport map** from $p$ to $q$ in 1D is:

$$T^*(r) = F_q^{-1}(F_p(r))$$

**Histogram equalization is optimal transport to uniform:** Setting $q = \text{Uniform}[0,1]$ gives $F_q^{-1}(t) = t$, so:

$$T^*(r) = F_p(r) = \text{CDF}_p(r)$$

This is exactly histogram equalization! $\blacksquare$

### Step 4: Histogram Specification (Matching) via Optimal Transport

To match histogram $p$ to a target histogram $q$ (not necessarily uniform):

$$T^*(r) = F_q^{-1}(F_p(r))$$

This is a composition: equalize $p$ → then "de-equalize" to $q$.

**Proof this minimizes Wasserstein distance:**

Brenier's theorem (1D version): The unique monotone transport map minimizing $W_2$ is $T^* = F_q^{-1} \circ F_p$. Since CDFs are monotonically increasing, this map preserves order and is optimal. $\blacksquare$

### Step 5: Connection to RL Brightness/Contrast Control

Our RL agent applies $I' = \alpha I + \beta$, which is an **affine histogram transform**:

$$p'(s) = \frac{1}{\alpha} p\left(\frac{s - \beta}{\alpha}\right)$$

**Proof the mean shifts by $\beta$ and variance scales by $\alpha^2$:**

$\mu' = \alpha \mu + \beta$ (linear transform of mean)

$\sigma'^2 = E[(\alpha I + \beta - \alpha\mu - \beta)^2] = \alpha^2 E[(I - \mu)^2] = \alpha^2 \sigma^2$ $\blacksquare$

The agent's sequence of $(\alpha_t, \beta_t)$ actions progressively transforms the histogram toward the target distribution — performing piecewise-affine optimal transport.

## 4 — Gymnasium Environment

The environment wraps an image pair (degraded, target). The agent observes the current image state (as a tensor) and selects discrete brightness/contrast adjustment actions.

In [ ]:
class BrightnessContrastEnv(gym.Env):
    """RL environment for brightness/contrast correction."""

    metadata = {"render_modes": ["rgb_array"]}

    # 9 discrete actions: combinations of (Δα, Δβ)
    ACTION_MAP = [
        (1.05, 0.0),   # slight contrast increase
        (1.10, 0.0),   # moderate contrast increase
        (1.20, 0.0),   # strong contrast increase
        (0.95, 0.0),   # slight contrast decrease
        (1.0, 0.02),   # slight brightness increase
        (1.0, 0.05),   # moderate brightness increase
        (1.0, -0.02),  # slight brightness decrease
        (1.0, -0.05),  # moderate brightness decrease
        (1.0, 0.0),    # no-op
    ]

    def __init__(self, dataset, max_steps: int = 15, reward_weights=(1.0, 5.0)):
        super().__init__()
        self.dataset = dataset
        self.max_steps = max_steps
        self.lambda_psnr, self.lambda_ssim = reward_weights

        h, w, c = dataset[0]["target"].shape
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(c, h, w), dtype=np.float32)
        self.action_space = spaces.Discrete(len(self.ACTION_MAP))

        self.current_img = None
        self.target_img = None
        self.step_count = 0

    def _get_obs(self):
        return self.current_img.transpose(2, 0, 1)  # HWC → CHW

    def _compute_quality(self, img):
        psnr = compute_psnr(self.target_img, img, data_range=1.0)
        ssim = compute_ssim(self.target_img, img, data_range=1.0, channel_axis=2)
        return psnr, ssim

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = random.randint(0, len(self.dataset) - 1)
        self.target_img = self.dataset[idx]["target"].copy()
        self.current_img = self.dataset[idx]["degraded"].copy()
        self.step_count = 0
        self.prev_psnr, self.prev_ssim = self._compute_quality(self.current_img)
        return self._get_obs(), {"psnr": self.prev_psnr, "ssim": self.prev_ssim}

    def step(self, action: int):
        alpha, beta = self.ACTION_MAP[action]
        self.current_img = np.clip(alpha * self.current_img + beta, 0.0, 1.0).astype(np.float32)
        self.step_count += 1

        psnr, ssim = self._compute_quality(self.current_img)

        reward = (self.lambda_psnr * (psnr - self.prev_psnr) +
                  self.lambda_ssim * (ssim - self.prev_ssim))

        self.prev_psnr, self.prev_ssim = psnr, ssim
        terminated = self.step_count >= self.max_steps

        return self._get_obs(), float(reward), terminated, False, {"psnr": psnr, "ssim": ssim}


# Quick test
env = BrightnessContrastEnv(dataset)
obs, info = env.reset()
print(f"Observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print(f"Initial PSNR: {info['psnr']:.2f} dB, SSIM: {info['ssim']:.4f}")

## 5 — DQN Agent Architecture

The Q-network is a compact CNN that maps the image observation directly to Q-values for each discrete action.

In [ ]:
class QNetwork(nn.Module):
    """CNN-based Q-network for image enhancement."""

    def __init__(self, in_channels: int, num_actions: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.head = nn.Sequential(
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.head(x)


# Verify architecture
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
net = QNetwork(3, len(BrightnessContrastEnv.ACTION_MAP)).to(device)
print(f"Q-network output shape: {net(dummy).shape}")
print(f"Total parameters: {sum(p.numel() for p in net.parameters()):,}")

## Deep Derivation: Gamma Correction and Power Law Model

### Step 1: Gamma Correction Model

The display transfer function follows a power law:

$$L_{\text{display}} = V^{\gamma}$$

where $V$ is the input voltage (normalized pixel value) and $\gamma$ is the display gamma (typically $\gamma \approx 2.2$ for CRT monitors).

**Gamma correction** pre-compensates: $V_{\text{corrected}} = L^{1/\gamma}$

So the displayed luminance is: $L_{\text{display}} = (L^{1/\gamma})^{\gamma} = L$ (linear).

### Step 2: sRGB Gamma Curve (Piecewise)

The sRGB standard uses a piecewise function for perceptual uniformity:

$$V_{\text{sRGB}} = \begin{cases} 12.92 \cdot L & L \leq 0.0031308 \\ 1.055 \cdot L^{1/2.4} - 0.055 & L > 0.0031308 \end{cases}$$

**Why piecewise?** Pure power law has infinite slope at $L = 0$, amplifying noise in dark regions. The linear segment near zero prevents this.

**Proof of continuity at the break point:**

At $L = 0.0031308$:
- Linear: $12.92 \times 0.0031308 = 0.04045$
- Power: $1.055 \times 0.0031308^{1/2.4} - 0.055 = 1.055 \times 0.3928 - 0.055 = 0.04045$ ✓ $\blacksquare$

### Step 3: Optimal Gamma Estimation via Least Squares

Given degraded image $I_d$ and target $I^*$, estimate optimal gamma:

$$\hat{\gamma} = \arg\min_\gamma \sum_{i,j} (I_d(i,j)^\gamma - I^*(i,j))^2$$

Taking the log: $I_d^\gamma = e^{\gamma \ln I_d}$. The gradient:

$$\frac{\partial \mathcal{L}}{\partial \gamma} = \sum_{i,j} 2(I_d^{\gamma} - I^*) \cdot I_d^{\gamma} \ln I_d$$

Setting to zero gives a transcendental equation — solved iteratively (Newton's method) or by grid search.

### Step 4: Weber's Law and Perceptual Brightness

**Weber's Law:** The just-noticeable difference (JND) in luminance is proportional to background luminance:

$$\frac{\Delta L}{L} = k_W \approx 0.02$$

This implies logarithmic perception: $\text{perceived brightness} \propto \log L$.

**Consequence for RL:** Equal-sized actions $\Delta\beta$ produce different perceptual effects at different brightness levels. Multiplicative actions $\alpha \cdot I$ are more perceptually uniform — which is why our action space includes contrast multipliers.

### Step 5: DQN Loss Analysis for Enhancement

The Huber loss used in our DQN:

$$\mathcal{L}_\delta(u) = \begin{cases} \frac{1}{2}u^2 & |u| \leq \delta \\ \delta(|u| - \frac{1}{2}\delta) & |u| > \delta \end{cases}$$

**Proof Huber loss is continuously differentiable:**

At $|u| = \delta$: $\frac{1}{2}\delta^2 = \delta(\delta - \frac{1}{2}\delta) = \frac{1}{2}\delta^2$ ✓

Derivative at $|u| = \delta$: $\frac{\partial}{\partial u}\frac{1}{2}u^2\big|_{u=\delta} = \delta$ and $\frac{\partial}{\partial u}\delta(|u| - \frac{1}{2}\delta)\big|_{u=\delta} = \delta$ ✓

**Advantage over MSE:** Huber loss clips gradients for large TD errors $|u| > \delta$, preventing catastrophic updates when the Q-value estimate is far off — crucial for stable DQN training. $\blacksquare$

## 6 — Replay Buffer & DQN Training Loop

In [ ]:
class ReplayBuffer:
    """Fixed-size circular experience replay buffer."""

    def __init__(self, capacity: int):
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.LongTensor(actions).to(device),
            torch.FloatTensor(rewards).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(dones).to(device),
        )

    def __len__(self):
        return len(self.buffer)

In [ ]:
# Hyperparameters
NUM_EPISODES = 300
BATCH_SIZE = 64
GAMMA = 0.99
LR = 1e-3
BUFFER_SIZE = 10_000
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 200
TARGET_UPDATE = 10
MAX_STEPS = 15

# Networks
policy_net = QNetwork(3, env.action_space.n).to(device)
target_net = QNetwork(3, env.action_space.n).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
replay_buffer = ReplayBuffer(BUFFER_SIZE)


def select_action(state: np.ndarray, step_count: int) -> int:
    eps = EPS_END + (EPS_START - EPS_END) * math.exp(-step_count / EPS_DECAY)
    if random.random() < eps:
        return env.action_space.sample()
    with torch.no_grad():
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        return policy_net(state_t).argmax(dim=1).item()


def optimize_model():
    if len(replay_buffer) < BATCH_SIZE:
        return 0.0
    states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)

    q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        next_q = target_net(next_states).max(1)[0]
        target_q = rewards + GAMMA * next_q * (1 - dones)

    loss = F.smooth_l1_loss(q_values, target_q)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
    optimizer.step()
    return loss.item()

In [ ]:
# Training
episode_rewards = []
episode_psnrs = []
episode_ssims = []
losses = []
global_step = 0

for ep in range(NUM_EPISODES):
    obs, info = env.reset()
    ep_reward = 0.0

    for t in range(MAX_STEPS):
        action = select_action(obs, global_step)
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        replay_buffer.push(obs, action, reward, next_obs, float(done))
        obs = next_obs
        ep_reward += reward
        global_step += 1

        loss = optimize_model()
        if loss > 0:
            losses.append(loss)

        if done:
            break

    episode_rewards.append(ep_reward)
    episode_psnrs.append(info["psnr"])
    episode_ssims.append(info["ssim"])

    if ep % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if ep % 50 == 0:
        avg_r = np.mean(episode_rewards[-50:])
        avg_p = np.mean(episode_psnrs[-50:])
        print(f"Episode {ep:4d} | Avg Reward: {avg_r:7.2f} | Avg PSNR: {avg_p:6.2f} dB | ε: {EPS_END + (EPS_START - EPS_END) * math.exp(-global_step / EPS_DECAY):.3f}")

print("\nTraining complete!")

## 7 — Training Curves

In [ ]:
def moving_average(data, window=20):
    return np.convolve(data, np.ones(window)/window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(episode_rewards, alpha=0.3, color="steelblue")
axes[0].plot(moving_average(episode_rewards), color="navy", linewidth=2)
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Episode Reward")
axes[0].set_title("Training Reward")
axes[0].grid(True, alpha=0.3)

axes[1].plot(episode_psnrs, alpha=0.3, color="coral")
axes[1].plot(moving_average(episode_psnrs), color="darkred", linewidth=2)
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Final PSNR (dB)")
axes[1].set_title("PSNR over Training")
axes[1].grid(True, alpha=0.3)

axes[2].plot(losses[:500], alpha=0.3, color="mediumseagreen")
if len(losses) > 20:
    axes[2].plot(moving_average(losses[:500]), color="darkgreen", linewidth=2)
axes[2].set_xlabel("Optimisation Step")
axes[2].set_ylabel("Huber Loss")
axes[2].set_title("DQN Loss")
axes[2].grid(True, alpha=0.3)

plt.suptitle("Brightness/Contrast Agent — Training Dynamics", fontsize=14)
plt.tight_layout()
plt.show()

## Deep Derivation: Contrast Enhancement Theory and Automatic Exposure

### Step 1: Image Entropy as Quality Measure

Shannon entropy of the intensity histogram:

$$H(I) = -\sum_{k=0}^{L-1} p(k) \log_2 p(k)$$

where $p(k)$ is the probability of intensity level $k$. Maximum entropy $H_{\max} = \log_2 L$ occurs for uniform distribution (histogram equalization).

**Connection to contrast:** Low-contrast images have peaked histograms (low entropy). The RL agent's contrast-increasing actions spread the histogram, increasing entropy.

### Step 2: Contrast Ratio Metrics

**Michelson contrast:** $C_M = \frac{I_{\max} - I_{\min}}{I_{\max} + I_{\min}}$

**RMS contrast:** $C_{\text{RMS}} = \sqrt{\frac{1}{N}\sum_i (I_i - \bar{I})^2} = \sigma_I$

**Weber contrast (for pattern on background):** $C_W = \frac{I - I_b}{I_b}$

**Proof RMS contrast is invariant to brightness shift:**

$C_{\text{RMS}}(I + c) = \sqrt{\frac{1}{N}\sum_i (I_i + c - \bar{I} - c)^2} = \sqrt{\frac{1}{N}\sum_i (I_i - \bar{I})^2} = C_{\text{RMS}}(I) \quad \blacksquare$

But NOT invariant to scaling: $C_{\text{RMS}}(\alpha I) = \alpha \cdot C_{\text{RMS}}(I)$.

### Step 3: Automatic Exposure — Zone System Theory

Ansel Adams' Zone System maps scene luminance to print density across 11 zones (0-X):

$$\text{Zone} = 5 + \log_2\frac{L}{L_{\text{mid}}}$$

where $L_{\text{mid}}$ is the metered (18% gray) luminance.

**Exposure Value (EV):** $\text{EV} = \log_2\frac{N^2}{t}$ where $N$ = f-number, $t$ = shutter speed.

The RL agent's brightness control $\beta$ is analogous to **exposure compensation** ($\Delta\text{EV}$), while contrast control $\alpha$ adjusts the **tone curve slope** (zone compression/expansion).

### Step 4: Bellman Optimality for Sequential Enhancement

For our MDP with states $s_t = I_t$ (current image) and actions $a_t = (\alpha_t, \beta_t)$:

$$Q^*(s, a) = \underbrace{r(s, a)}_{\Delta\text{PSNR} + \lambda\Delta\text{SSIM}} + \gamma \max_{a'} Q^*(s', a')$$

**Insight:** The discount factor $\gamma = 0.99$ means the agent values future improvements almost as much as immediate ones. This encourages "setting up" good intermediate states — e.g., first correcting brightness (easy), then fine-tuning contrast (harder).

**Convergence guarantee:** With the target network update every $K$ steps and experience replay, DQN converges to $Q^*$ under mild conditions (finite action space, sufficient exploration). $\blacksquare$

## 8 — Evaluation: Before / After Visualisation

In [ ]:
def evaluate_agent(env, policy_net, num_episodes=6):
    """Run the trained agent and collect before/after images."""
    results = []
    policy_net.eval()

    for _ in range(num_episodes):
        obs, info = env.reset()
        initial_img = env.current_img.copy()
        initial_psnr = info["psnr"]

        for t in range(MAX_STEPS):
            with torch.no_grad():
                state_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
                action = policy_net(state_t).argmax(dim=1).item()
            obs, _, done, _, info = env.step(action)
            if done:
                break

        results.append({
            "degraded": initial_img,
            "enhanced": env.current_img.copy(),
            "target": env.target_img.copy(),
            "psnr_before": initial_psnr,
            "psnr_after": info["psnr"],
            "ssim_after": info["ssim"],
        })

    policy_net.train()
    return results

results = evaluate_agent(env, policy_net)

fig, axes = plt.subplots(3, 6, figsize=(20, 10))
for i, r in enumerate(results):
    axes[0, i].imshow(r["degraded"])
    axes[0, i].set_title(f"Degraded\nPSNR={r['psnr_before']:.1f}")
    axes[0, i].axis("off")

    axes[1, i].imshow(r["enhanced"])
    axes[1, i].set_title(f"Enhanced\nPSNR={r['psnr_after']:.1f}")
    axes[1, i].axis("off")

    axes[2, i].imshow(r["target"])
    axes[2, i].set_title("Target")
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Degraded", fontsize=12)
axes[1, 0].set_ylabel("Agent Output", fontsize=12)
axes[2, 0].set_ylabel("Ground Truth", fontsize=12)
plt.suptitle("Brightness/Contrast Agent — Before vs After Enhancement", fontsize=14)
plt.tight_layout()
plt.show()

print("\nNumerical Results:")
print(f"{'Image':>6} | {'PSNR Before':>12} | {'PSNR After':>12} | {'ΔPSNR':>8} | {'SSIM After':>10}")
print("-" * 60)
for i, r in enumerate(results):
    delta = r['psnr_after'] - r['psnr_before']
    print(f"{i:6d} | {r['psnr_before']:12.2f} | {r['psnr_after']:12.2f} | {delta:+8.2f} | {r['ssim_after']:10.4f}")

## 9 — Analysing Learned Action Sequences

In [ ]:
def trace_episode(env, policy_net):
    """Record the full action sequence and PSNR trajectory of one episode."""
    obs, info = env.reset()
    psnr_traj = [info["psnr"]]
    actions_taken = []
    policy_net.eval()

    for t in range(MAX_STEPS):
        with torch.no_grad():
            state_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            action = policy_net(state_t).argmax(dim=1).item()
        actions_taken.append(action)
        obs, _, done, _, info = env.step(action)
        psnr_traj.append(info["psnr"])
        if done:
            break

    policy_net.train()
    return psnr_traj, actions_taken


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for trial in range(5):
    psnr_traj, actions = trace_episode(env, policy_net)
    ax1.plot(psnr_traj, marker="o", markersize=4, label=f"Trial {trial}")

ax1.set_xlabel("Step")
ax1.set_ylabel("PSNR (dB)")
ax1.set_title("PSNR Trajectory During Episode")
ax1.legend()
ax1.grid(True, alpha=0.3)

action_labels = ["α+5%", "α+10%", "α+20%", "α-5%", "β+.02", "β+.05", "β-.02", "β-.05", "noop"]
all_actions = []
for _ in range(50):
    _, acts = trace_episode(env, policy_net)
    all_actions.extend(acts)

counts = np.bincount(all_actions, minlength=len(action_labels))
ax2.barh(action_labels, counts, color="steelblue")
ax2.set_xlabel("Frequency")
ax2.set_title("Action Distribution (50 Episodes)")
ax2.grid(True, alpha=0.3, axis="x")

plt.suptitle("Learned Behaviour Analysis", fontsize=14)
plt.tight_layout()
plt.show()

## 10 — Key Takeaways

| Insight | Detail |
|---------|--------|
| **RL formulation** | Brightness/contrast adjustment naturally maps to a sequential MDP |
| **Action design** | Discrete multiplicative (contrast) and additive (brightness) deltas give the agent fine-grained control |
| **Reward shaping** | Combining PSNR and SSIM improvements guides the agent to restore both fidelity and structure |
| **DQN suitability** | With a small discrete action space, vanilla DQN converges reliably |
| **Limitations** | Global adjustments only — spatially-varying degradation would require policy-per-patch or actor-critic methods |

---

*Next notebook → Module 7.2: Color Correction Agent using RL with HSV/LAB colour space operations.*